In [ ]:
!pip install langchain_community langchainhub chromadb langchain langchain-groq langchain-text-splitters sentence-transformers

In [ ]:
import os
os.environ['GROQ_API_KEY'] = "YOURAPIKEY"  # Replace with your Groq API key
# Get a free API key at https://console.groq.com

In [ ]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader(web_paths=["https://en.wikipedia.org/wiki/One_Piece"])

docs = loader.load()
print(docs)

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)
splits = text_splitter.split_documents(docs)

In [ ]:
print(splits[0])
print(splits[1])
print(splits[2])

In [ ]:
print(len(splits))

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# Using a local sentence-transformers model for embeddings (Groq doesn't provide embeddings)
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(documents=splits, embedding=embeddings)

In [ ]:
print(vectorstore._collection.count())

In [ ]:
print(vectorstore._collection.get())

In [ ]:
print("\nCollection 1 - ", vectorstore._collection.get(ids=['28651d9a-ab51-41f8-ab83-e68285623c4e'], include=["embeddings", "documents"]))
print("\nCollection 2 - ", vectorstore._collection.get(ids=['054dee19-19ed-4574-bc51-511060fd707a'], include=["embeddings", "documents"]))
print("\nCollection 3 - ", vectorstore._collection.get(ids=['2fd71cb4-835a-43c5-b920-b7e1be51c450'], include=["embeddings", "documents"]))

In [ ]:
retriever = vectorstore.as_retriever()

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""You are an assistant for question-answering tasks. \
Use the following pieces of retrieved context to answer the question. \
If you don't know the answer, just say that you don't know. \
Use three sentences maximum and keep the answer concise.

Question: {question}
Context: {context}
Answer:""")

In [ ]:
from langchain_groq import ChatGroq
llm = ChatGroq(model="llama-3.1-8b-instant")  # Alternatives: "llama-3.3-70b-versatile", "gemma2-9b-it"

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [ ]:
def format_docs(docs):
  return "\n".join(doc.page_content for doc in docs)

In [ ]:
rag_chain = ({"context" : retriever | format_docs, "question" : RunnablePassthrough()}
             | prompt
             | llm
             | StrOutputParser())

In [ ]:
rag_chain.invoke("What is One piece")

In [ ]:
rag_chain.invoke("Author of One piece")

In [ ]:
rag_chain.invoke("Total sales in manga")

In [ ]:
rag_chain.invoke("Who is Monkey D Luffy")

In [ ]:
from langchain_core.runnables import RunnableLambda

In [ ]:
def print_prompt(prompt_text):
  print("Prompt - ", prompt_text)
  return prompt_text

In [ ]:
rag_chain_with_print = ({"context" : retriever | format_docs, "question" : RunnablePassthrough()}
             | prompt
             | RunnableLambda(print_prompt)
             | llm
             | StrOutputParser())

In [ ]:
rag_chain_with_print.invoke("Who is Zoro")